# 8 · DSPy — compile your prompts
Declare I/O + a strategy; an optimizer writes the few-shot demos.

In [ ]:
# Bootstrap: make the repo root importable so `import config` works from notebooks/
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from config import assert_key
assert_key()
print("Gateway ready.")

In [ ]:
import dspy
from config import SMART_MODEL_LITELLM, API_KEY, BASE_URL
dspy.configure(lm=dspy.LM(SMART_MODEL_LITELLM, api_key=API_KEY, api_base=BASE_URL))

class QA(dspy.Signature):
    'Answer with ONLY the final number, no words.'
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()

program = dspy.ChainOfThought(QA)
print("before:", program(question="A robot travels 12 km in 4 hours. Speed in km/h?").answer)

### Optimize against a metric (makes several calls)

In [ ]:
trainset = [
    dspy.Example(question="6 units at $3 each. Total?", answer="18").with_inputs("question"),
    dspy.Example(question="A tank holds 20L, 5L used. Left?", answer="15").with_inputs("question"),
    dspy.Example(question="3 robots, 4 arms each. Arms?", answer="12").with_inputs("question"),
]
metric = lambda ex, pred, trace=None: ex.answer.strip() == pred.answer.strip()
optimized = dspy.BootstrapFewShot(metric=metric, max_bootstrapped_demos=2).compile(
    dspy.ChainOfThought(QA), trainset=trainset)
print("after:", optimized(question="A robot travels 12 km in 4 hours. Speed in km/h?").answer)
dspy.inspect_history(n=1)   # see the auto-generated few-shot demos

## 🧪 Your turn
1. Add 2–3 more `trainset` examples.
2. Loosen the `metric` to allow ±1 tolerance (parse ints).
3. Re-compile and compare the `inspect_history` prompt — did the demos change?

In [ ]:
# your code here